# Focal — training on a GPU

Runs the full training schedule: 10 epochs with the backbone frozen, then 15 fine-tuning the last two blocks, on the whole corpus.

On a free Colab T4 this takes roughly 20-30 minutes. The same schedule on an 8-core CPU takes about ten hours, which is why local runs use cached embeddings and a fine-tuning subsample.

## Quick Start
1. **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells top to bottom.
3. For the corpus (`bundle.zip` ~740 MB), either mount Google Drive with `bundle.zip` or upload it directly.

## 1. Check the GPU

If this prints `CUDA: False`, go to **Runtime -> Change runtime type -> T4 GPU** before continuing.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY - switch runtime type")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true

## 2. Clone Code & Install Dependencies

Clones the repository and installs the package in editable mode with training dependencies.

In [ ]:
REPO = "https://github.com/SreeAditya-Dev/Focal.git"

import pathlib, os

if not pathlib.Path("pyproject.toml").exists():
    if not pathlib.Path("ml_pipeline/pyproject.toml").exists():
        if REPO:
            !git clone -q $REPO repo && cp -r repo/ml_pipeline .
        else:
            from google.colab import files
            import zipfile
            uploaded = files.upload()   # zip of local ml_pipeline folder
            with zipfile.ZipFile(next(iter(uploaded))) as archive:
                archive.extractall(".")
    if pathlib.Path("ml_pipeline").is_dir():
        %cd ml_pipeline

!pip install -q -e ".[train]"
print("Package installed successfully. Working directory:", os.getcwd())

## 3. Load the Training Dataset Bundle

The bundle contains 224px images and the full-resolution `features.parquet` table.
Upload `bundle.zip` (built via `python -m training.export_bundle --zip`) to Google Drive (e.g. `MyDrive/bundle.zip` or `MyDrive/focal/bundle.zip`) or directly to Colab.

In [ ]:
USE_DRIVE = True
DRIVE_PATH = "/content/drive/MyDrive/bundle.zip"  # <- Change path if stored elsewhere in Drive

import pathlib, zipfile, os

dest_dir = pathlib.Path("dataset/generated")
dest_dir.mkdir(parents=True, exist_ok=True)

if not (dest_dir / "features.parquet").exists():
    candidates = [
        pathlib.Path(DRIVE_PATH),
        pathlib.Path("/content/drive/MyDrive/focal/bundle.zip"),
        pathlib.Path("/content/bundle.zip"),
        pathlib.Path("/content/ml_pipeline/dataset/bundle.zip"),
        pathlib.Path("dataset/bundle.zip"),
    ]
    src = next((str(p) for p in candidates if p.exists()), None)

    if src is None:
        if USE_DRIVE:
            from google.colab import drive
            drive.mount("/content/drive")
            # Re-check drive candidates after mount
            src = next((str(p) for p in candidates if p.exists()), DRIVE_PATH)
        else:
            from google.colab import files
            print("Upload bundle.zip:")
            uploaded = files.upload()
            src = next(iter(uploaded))

    print(f"Extracting bundle from {src} to {dest_dir}...")
    with zipfile.ZipFile(src) as archive:
        archive.extractall(dest_dir)
    print("Extraction complete.")

import pandas as pd
manifest = pd.read_csv('dataset/generated/manifest.csv')
features = pd.read_parquet('dataset/generated/features.parquet')
print(f"{len(manifest)} images, {len(features)} feature rows")
print(manifest.split.value_counts().to_string())
assert len(features) >= len(manifest) * 0.95, 'feature table is incomplete'

## 4. Fit Rule Thresholds

Refits the deterministic CV rule thresholds to percentiles of the training split.

In [ ]:
!python -m training.fit_rules --out models/rules_v1.json

## 5. Train Hybrid Model (CNN + Classical Features)

Runs full training schedule on GPU: 10 head epochs followed by 15 fine-tuning epochs.

In [ ]:
!python -m training.train \
  --head-epochs 10 \
  --finetune-epochs 15 \
  --batch-size 64 \
  --num-workers 2

## 6. Train Ablation Variants

Trains Image-only CNN and Features-only MLP baseline models for ablation comparisons.

In [ ]:
!python -m training.train --ablation image    --head-epochs 10 --finetune-epochs 15 --batch-size 64 --num-workers 2
!python -m training.train --ablation features --head-epochs 30 --batch-size 256

## 7. Calibrate & Tune Fusion

Fits temperature scaling calibration parameters and tunes optimal fusion weights on the validation split.

In [ ]:
!python -m training.calibrate \
  --model models/focal_cnn_v1.pt \
  --rules models/rules_v1.json \
  --tune-fusion

## 8. Evaluate on Held-Out Test Split

Evaluates hybrid model, ablation variants, and rules-only baseline on test data.

In [ ]:
!python -m evaluation.evaluate --ablation

In [ ]:
from IPython.display import Image, display
if pathlib.Path("evaluation/reports/reliability.png").exists():
    display(Image("evaluation/reports/reliability.png"))
if pathlib.Path("evaluation/reports/precision_recall.png").exists():
    display(Image("evaluation/reports/precision_recall.png"))

## 9. Export & Download Trained Artefacts

Zips the trained models and evaluation reports. Copy `focal_models.zip` into `ml_pipeline/models/` for serving in the backend API.

In [ ]:
import shutil, os
try:
    from google.colab import files
    has_colab = True
except ImportError:
    has_colab = False

shutil.make_archive("focal_models", "zip", "models")
shutil.make_archive("focal_reports", "zip", "evaluation/reports")
print("Created focal_models.zip and focal_reports.zip in", os.getcwd())

if has_colab:
    try:
        files.download("focal_models.zip")
        files.download("focal_reports.zip")
    except Exception as e:
        print("Colab automatic download skipped:", e)